# ASL Gesture Recognition — Data Cleaning
**Run this in WSL Ubuntu with asl_env activated**

```bash
cd /mnt/c/HiASL_BSE_FYP_2026/asl-gesture-recognition-model
source asl_env/bin/activate
jupyter notebook
```

### What this notebook does:
1. Load all `.npy` files from `static/dataset/`
2. Verify shapes — remove any corrupted files
3. Remove statistical outliers per class
4. Cap classes at 600 samples (Option D)
5. Report final class balance
6. Save cleaned dataset as `X_clean.npy` and `y_clean.npy`

## 1. Setup & Imports

In [ ]:
import numpy as np
import os
import random
import matplotlib.pyplot as plt
from collections import Counter

# ── Configuration ────────────────────────────────────────────
DATASET_DIR  = '/mnt/c/HiASL_BSE_FYP_2026/asl-gesture-recognition-model/static/dataset'
OUTPUT_DIR   = '/mnt/c/HiASL_BSE_FYP_2026/asl-gesture-recognition-model/static'
CAP_SAMPLES  = 600       # max samples per class (Option D)
OUTLIER_STD  = 3.0       # z-score threshold for outlier removal
RANDOM_SEED  = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# All 36 classes
ALL_CLASSES = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ') + [str(i) for i in range(10)]
print(f'Classes: {ALL_CLASSES}')
print(f'Total classes: {len(ALL_CLASSES)}')
print(f'Dataset directory: {DATASET_DIR}')

## 2. Load All .npy Files

In [ ]:
X_raw = []   # landmark arrays
y_raw = []   # class labels (string)

load_report = {}
shape_errors = 0

for class_name in ALL_CLASSES:
    class_dir = os.path.join(DATASET_DIR, class_name)
    
    if not os.path.exists(class_dir):
        print(f'[WARN] Missing folder: {class_name}')
        load_report[class_name] = 0
        continue
    
    files = [f for f in os.listdir(class_dir) if f.endswith('.npy')]
    loaded = 0
    
    for f in files:
        try:
            arr = np.load(os.path.join(class_dir, f))
            
            # Shape check — must be exactly (63,)
            if arr.shape != (63,):
                shape_errors += 1
                continue
            
            # NaN/Inf check
            if not np.isfinite(arr).all():
                shape_errors += 1
                continue
                
            X_raw.append(arr)
            y_raw.append(class_name)
            loaded += 1
            
        except Exception as e:
            shape_errors += 1
    
    load_report[class_name] = loaded

X_raw = np.array(X_raw)   # shape: (N, 63)
y_raw = np.array(y_raw)   # shape: (N,)

print(f'\nLoaded: {len(X_raw)} samples')
print(f'Shape errors / corrupted files skipped: {shape_errors}')
print(f'\nPer-class counts after loading:')
for cls, count in load_report.items():
    print(f'  {cls}: {count}')

## 3. Outlier Removal

In [ ]:
X_no_outliers = []
y_no_outliers = []
outlier_report = {}

for class_name in ALL_CLASSES:
    mask = y_raw == class_name
    X_class = X_raw[mask]  # all samples for this class
    
    if len(X_class) == 0:
        outlier_report[class_name] = {'before': 0, 'removed': 0, 'after': 0}
        continue
    
    # Calculate mean and std per feature across all samples in class
    mean = X_class.mean(axis=0)   # shape: (63,)
    std  = X_class.std(axis=0)    # shape: (63,)
    std[std == 0] = 1e-6          # avoid division by zero
    
    # Z-score for each sample: max z-score across all 63 features
    z_scores = np.abs((X_class - mean) / std)  # shape: (N, 63)
    max_z    = z_scores.max(axis=1)             # shape: (N,)
    
    # Keep samples where max z-score is within threshold
    keep_mask = max_z <= OUTLIER_STD
    X_kept = X_class[keep_mask]
    
    removed = len(X_class) - len(X_kept)
    outlier_report[class_name] = {
        'before': len(X_class),
        'removed': removed,
        'after': len(X_kept)
    }
    
    for sample in X_kept:
        X_no_outliers.append(sample)
        y_no_outliers.append(class_name)

X_no_outliers = np.array(X_no_outliers)
y_no_outliers = np.array(y_no_outliers)

print('Outlier Removal Report:')
print(f'{"Class":<8} {"Before":>7} {"Removed":>8} {"After":>6}')
print('-' * 35)
total_removed = 0
for cls, r in outlier_report.items():
    total_removed += r['removed']
    flag = ' <- check' if r['removed'] > r['before'] * 0.2 else ''
    print(f'{cls:<8} {r["before"]:>7} {r["removed"]:>8} {r["after"]:>6}{flag}')
print('-' * 35)
print(f'Total outliers removed: {total_removed}')
print(f'Remaining samples: {len(X_no_outliers)}')

## 4. Cap Classes at 600 Samples (Option D)

In [ ]:
X_capped = []
y_capped = []
cap_report = {}

for class_name in ALL_CLASSES:
    mask    = y_no_outliers == class_name
    X_class = X_no_outliers[mask]
    n       = len(X_class)
    
    if n == 0:
        cap_report[class_name] = {'original': 0, 'final': 0}
        continue
    
    if n > CAP_SAMPLES:
        # Randomly select CAP_SAMPLES
        indices  = np.random.choice(n, CAP_SAMPLES, replace=False)
        X_select = X_class[indices]
    else:
        # Keep all — below cap
        X_select = X_class
    
    cap_report[class_name] = {'original': n, 'final': len(X_select)}
    
    for sample in X_select:
        X_capped.append(sample)
        y_capped.append(class_name)

X_capped = np.array(X_capped)
y_capped = np.array(y_capped)

print(f'Cap set to: {CAP_SAMPLES} samples per class')
print(f'\nClass balance after capping:')
print(f'{"Class":<8} {"Before":>7} {"Final":>6} {"Status":>10}')
print('-' * 38)
for cls, r in cap_report.items():
    status = 'capped' if r['original'] > CAP_SAMPLES else 'kept all'
    low    = ' <- LOW' if r['final'] < 400 else ''
    print(f'{cls:<8} {r["original"]:>7} {r["final"]:>6} {status:>10}{low}')
print('-' * 38)
print(f'Total samples: {len(X_capped)}')

## 5. Visualize Final Class Distribution

In [ ]:
counts = Counter(y_capped)
classes = ALL_CLASSES
values  = [counts.get(c, 0) for c in classes]

colors = ['#e74c3c' if v < 400 else '#f39c12' if v < 500 else '#2ecc71' for v in values]

plt.figure(figsize=(16, 5))
bars = plt.bar(classes, values, color=colors, edgecolor='white', linewidth=0.5)
plt.axhline(y=CAP_SAMPLES, color='blue', linestyle='--', alpha=0.5, label=f'Cap ({CAP_SAMPLES})')
plt.axhline(y=400, color='red', linestyle='--', alpha=0.5, label='Low threshold (400)')
plt.title('Final Class Distribution After Cleaning', fontsize=14, fontweight='bold')
plt.xlabel('Class')
plt.ylabel('Sample Count')
plt.legend()
plt.tight_layout()

# Add count labels on bars
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(val), ha='center', va='bottom', fontsize=7)

plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as class_distribution.png')

## 6. Final Verification

In [ ]:
print('=== FINAL DATASET VERIFICATION ===')
print(f'X shape: {X_capped.shape}')        # should be (N, 63)
print(f'y shape: {y_capped.shape}')        # should be (N,)
print(f'Total samples: {len(X_capped)}')
print(f'Total classes: {len(np.unique(y_capped))}')
print(f'Feature dimensions: {X_capped.shape[1]}')  # should be 63
print(f'NaN values: {np.isnan(X_capped).sum()}')
print(f'Inf values: {np.isinf(X_capped).sum()}')
print(f'Min value: {X_capped.min():.4f}')
print(f'Max value: {X_capped.max():.4f}')
print(f'\nClass counts:')
for cls in ALL_CLASSES:
    count = (y_capped == cls).sum()
    bar   = '█' * (count // 10)
    print(f'  {cls}: {count:>4}  {bar}')

## 7. Save Cleaned Dataset

In [ ]:
X_path = os.path.join(OUTPUT_DIR, 'X_clean.npy')
y_path = os.path.join(OUTPUT_DIR, 'y_clean.npy')

np.save(X_path, X_capped)
np.save(y_path, y_capped)

print('Cleaned dataset saved:')
print(f'  X_clean.npy → {X_path}')
print(f'  y_clean.npy → {y_path}')
print(f'\nX_clean shape: {X_capped.shape}')
print(f'y_clean shape: {y_capped.shape}')
print(f'\nNext step: Run mlp_training.ipynb in WSL.')